In [70]:
!pip install langchain langchain-community langchain-google-genai pageindex rank-bm25 --quiet
# !: for running the commands directly to the shell not as python code
# --quiet : for not showing the logs

In [71]:
!pip install nltk --quiet

In [72]:
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

In [73]:
corpus_urls = np.array([
    'https://www.theguardian.com/books/2026/jun/02/classic-novels-relearn-how-to-read-distractions-screens',
    "https://www.theguardian.com/world/2026/jun/02/tuesday-briefing-palantirs-rise-and-why-so-many-oppose-its-role-in-the-british-state",
    "https://www.theguardian.com/commentisfree/2026/jun/02/british-politics-ideas-for-the-future-labour-policy-tony-blair",
    "https://www.theguardian.com/books/2026/jun/02/my-only-boy-by-rosa-rankin-gee-review-a-darkly-funny-near-future-dystopia",
    "https://www.theguardian.com/technology/2026/jun/02/google-alphabet-sell-stock-ai-share-sale-berkshire-hathaway"
])
# storing all the links in the array for extracting the data

In [74]:
import re
import nltk
from nltk.corpus import stopwords

try:
    STOP_WORDS = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords')
    STOP_WORDS = set(stopwords.words('english'))


def clean_text(text: str) -> str:
    if not text:
        return ""

    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()
    words = [word for word in words if word not in STOP_WORDS]

    return " ".join(words)

In [75]:
full_corpus = ""

for url in corpus_urls:
    response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

    articles = ""

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")
        para = soup.find_all("p")

        for p in para:
            articles += p.get_text()

        article = clean_text(articles)
        full_corpus += article + "\n"

In [76]:
corpus_filename = "cleaned_corpus.txt"
with open(corpus_filename, "w", encoding="utf-8") as f:
    f.write(full_corpus)
print(f"Saved preprocessed corpus to '{corpus_filename}'")

Saved preprocessed corpus to 'cleaned_corpus.txt'


In [77]:
!pip install -U langchain-text-splitters

In [78]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter tries to split on paragraphs first (\n\n),
# then sentences (\n), then words — preserving as much semantic meaning as possible.
# chunk_size=500: each chunk is at most 500 characters, keeping chunks small enough
# for BM25 to retrieve meaningfully specific passages rather than huge blobs.
# chunk_overlap=50: the last 50 characters of one chunk are repeated at the start
# of the next, so a sentence that falls on a boundary is not lost from either chunk.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Read back the cleaned corpus from disk (the same file saved in the previous cell)
# so that chunking always operates on the preprocessed text, not raw scraped text.
with open(corpus_filename, "r", encoding="utf-8") as f:
    cleaned_corpus_text = f.read()

# Split the full corpus into a list of smaller string chunks
corpus_chunks = text_splitter.split_text(cleaned_corpus_text)

# Overwrite cleaned_corpus.txt with the chunks joined by double newlines.
# This is the file you will upload to PageIndex in the next step.
with open(corpus_filename, "w", encoding="utf-8") as f:
    f.write("\n\n".join(corpus_chunks))

print(f"Total chunks created: {len(corpus_chunks)}")
print(f"Sample chunk:\n{corpus_chunks[0]}")

Total chunks created: 69
Sample chunk:
less decade surrounded screens lost ability read best books ever written inspired guardian 100 best novels list determined get backit privilege surrounded books parents hail literary working class subsection society believes great works lead richer life reading inverted form class snobbery dad could read well anyone prove package holidays sitting balcony entire time head bowed cigarette hand flicking pages jane austen herman melville difference old man old etonian drudgery employment paraphrase


In [79]:
! pip install reportlab

In [80]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

# Read the chunked corpus that was saved in the previous step
with open(corpus_filename, "r", encoding="utf-8") as f:
    cleaned_corpus_text = f.read()

# Split back into individual chunks on the double newline separator
chunks = cleaned_corpus_text.split("\n\n")

pdf_filename = "cleaned_corpus.pdf"

# SimpleDocTemplate handles page breaks and margins automatically
doc = SimpleDocTemplate(pdf_filename, pagesize=letter)
styles = getSampleStyleSheet()
story = []

for chunk in chunks:
    chunk = chunk.strip()
    if not chunk:
        continue

    # Each chunk becomes a Paragraph — reportlab handles line wrapping
    # and flowing text across pages automatically
    story.append(Paragraph(chunk, styles["Normal"]))

    # Small vertical gap between chunks so PageIndex can distinguish
    # them as separate passages when it parses the PDF structure
    story.append(Spacer(1, 8))

doc.build(story)
print(f"PDF saved as '{pdf_filename}' with {len(chunks)} chunks.")

PDF saved as 'cleaned_corpus.pdf' with 69 chunks.


In [81]:
import os
import time
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [82]:
try:
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    PAGEINDEX_API_KEY = userdata.get("PAGEINDEX")

except:
    print("Running outside Colab")

In [83]:
! pip install pageindex openai

In [84]:
import os
import json
import asyncio
from pageindex import PageIndexClient
from openai import AsyncOpenAI

In [85]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
pi_client = PageIndexClient(api_key=userdata.get('PAGEINDEX'))

In [86]:
import pageindex.utils as utils
doc = pi_client.submit_document("cleaned_corpus.pdf")
doc_id = doc["doc_id"]
# Tree generation takes a bit, so we poll
while not pi_client.is_retrieval_ready(doc_id):
    print("Still indexing...")
    import time; time.sleep(5)
# Grab the tree and take a look
tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
print("Document Tree:")
utils.print_tree(tree)

Still indexing...
Still indexing...
Still indexing...
Document Tree:
[{'title': 'Page 1', 'node_id': '0000', 'summary': "The text explores the author's struggle ..."},
 {'title': 'Page 2', 'node_id': '0001', 'summary': 'The text explores the difficulties moder...'},
 {'title': 'Page 3', 'node_id': '0002', 'summary': 'The text combines a personal reflection ...'},
 {'title': 'Page 4', 'node_id': '0003', 'summary': 'This text examines Palantir, a powerful ...'},
 {'title': 'Page 5', 'node_id': '0004', 'summary': 'This text provides a diverse roundup of ...'},
 {'title': 'Page 6', 'node_id': '0005', 'summary': 'The text evaluates the legacy of Tony Bl...'},
 {'title': 'Page 7', 'node_id': '0006', 'summary': 'The text combines a literary review of R...'},
 {'title': 'Page 8', 'node_id': '0007', 'summary': 'Alphabet is undertaking an unprecedented...'}]


In [87]:
from langchain_core.documents import Document

extracted_documents = []

def traverse_pageindex_tree(pages):
    """
    Processes the flat list of page objects returned by PageIndex's get_tree().
    Each object has a 'title', 'node_id', and 'summary' — there are no nested
    children to recurse into, so a simple loop replaces the old recursive approach.

    We use 'summary' as the page_content because that is the text PageIndex
    extracted and condensed from each page of the PDF — it is what BM25 will
    match queries against.

    'node_id' is stored in metadata so we can trace any retrieved chunk back
    to its exact page in the original PDF if needed.
    """
    for page in pages:
        summary = page.get("summary", "").strip()
        if not summary:
            # Skip pages where PageIndex returned an empty summary
            continue

        clean_summary = clean_text(summary)

        doc = Document(
            page_content=clean_summary,
            metadata={
                "source": corpus_filename,
                "page_title": page.get("title", ""),
                "node_id": page.get("node_id", "")
            }
        )
        extracted_documents.append(doc)

    print(f"Extracted {len(extracted_documents)} page summaries from PageIndex tree.")


# Pass the real tree returned by pi_client.get_tree(doc_id)
traverse_pageindex_tree(tree)

Extracted 8 page summaries from PageIndex tree.


In [88]:
from langchain_core.retrievers import BaseRetriever
from rank_bm25 import BM25Okapi
from pydantic import Field
from typing import List

class BM25Retriever(BaseRetriever):

    docs: List[Document] = Field(default_factory=list)
    bm25: BM25Okapi = Field(default=None)

    def _get_relevant_documents(self, query: str):

        query_tokens = query.lower().split()

        return self.bm25.get_top_n(
            query_tokens,
            self.docs,
            n=3
        )

In [89]:
tokenized_corpus = [
    doc.page_content.lower().split()
    for doc in extracted_documents
]

bm25_index = BM25Okapi(tokenized_corpus)

bm25_retriever = BM25Retriever(
    docs=extracted_documents,
    bm25=bm25_index
)

print(len(extracted_documents))
print("Retriever Created")

8
Retriever Created


In [90]:
! pip install langchain_classic

In [91]:
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0.0
# )

# system_prompt = (
#     "You are an assistant answering questions strictly based on the provided context.\n"
#     "If the answer is not contained within the context, state 'Information not found.'\n\n"
#     "Context:\n{context}"
# )

# prompt_template = ChatPromptTemplate.from_messages([
#     ("system", system_prompt),
#     ("human", "{input}")
# ])
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)

# The prompt template must use {context} and {input} exactly —
# create_stuff_documents_chain injects retrieved docs into {context}
# and create_retrieval_chain passes the user query into {input}
system_prompt = (
    "You are an assistant answering questions strictly based on the provided context.\n"
    "If the answer is not contained within the context, state 'Information not found.'\n\n"
    "Context:\n{context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# Step 1 — document chain: takes the retrieved docs, stuffs them all
# into the {context} slot of the prompt, then passes it to the LLM
document_chain = create_stuff_documents_chain(llm, prompt_template)

# Step 2 — retrieval chain: wraps around the document chain.
# When invoked, it first calls bm25_retriever to fetch relevant docs,
# then feeds them into the document chain above
retrieval_chain = create_retrieval_chain(bm25_retriever, document_chain)

print("Retrieval chain built successfully.")

Retrieval chain built successfully.


In [98]:
import time

query_test = "Palantir’s listing on the US stock market in 2020 saw its value increase by more than 1,500%, as excitement about artificial intelligence and widespread adoption by government agencies globally radically sped up its growth."
# --- BM25 RAG via LangChain chain ---
# retrieval_chain.invoke handles retrieval + prompt formatting + LLM call
# in one step. The response dict contains 'answer' and 'context' keys.
start_rag = time.time()

rag_result = retrieval_chain.invoke({"input": query_test})

end_rag = time.time()
rag_latency = end_rag - start_rag

# --- Naive full-corpus approach ---
start_naive = time.time()

naive_prompt = f"""Answer strictly using the corpus.

Question:
{query_test}

Corpus:
{full_corpus}
"""
naive_response = llm.invoke(naive_prompt)

end_naive = time.time()
naive_latency = end_naive - start_naive

# --- Print comparison ---
print("=" * 50)
print("BM25 RAG LATENCY")
print(rag_latency)
print("\nBM25 RAG ANSWER")
print(rag_result["answer"])

print("\n" + "=" * 50)
print("NAIVE LATENCY")
print(naive_latency)
print("\nNAIVE ANSWER")
print(naive_response.content)
print("=" * 50)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 11.149270628s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

BM25 RAG LATENCY
2.376581907272339

BM25 RAG ANSWER
The provided context does not contain information about why screens have altered the way we live. It discusses Alphabet's fundraising for AI infrastructure, the AI arms race, market concerns about high expenditures, and a literary review of a novel.

NAIVE LATENCY
4.3195436000823975

NAIVE ANSWER
Screens have altered the way we live by promoting a shallower reading experience, encouraging skimming and scanning, and undermining reading in general. Our dependence on screens has led to a form of text fatigue. According to research psychologist Gloria Mark, screens compel us to switch attention, pushing us towards new shiny things and making us focus on interfaces, ads, and dialogic elements rather than content. This is reflected in research showing that one in three readers spend less than 15 seconds on a given article. Furthermore, people spend their days staring at screens, drowning in instant messages and emails for work, which affect